# inferchaos on a T4: kill a real vLLM replica mid-load, behind llmgateway, and measure recovery

Qwen2.5-1.5B-Instruct on vLLM as the first provider; a tokenbench mock as the fallback. Steady load through the gateway; after a baseline the vLLM process is killed (`SIGKILL`), restarted after 60 s, and the run continues until the breaker closes again. Scored with the same SLO arithmetic as the mock experiments. ~10 min install, ~8 min run (the restart includes vLLM's full cold start).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# vLLM on the Kaggle image: the cu129 release wheel with the matching PyTorch index (the default wheel targets
# CUDA 13), and the image's preinstalled torchaudio/torchcodec removed because they are built for other CUDA
# versions and vLLM imports them (torchcodec is re-pulled by vLLM's requirements, hence the second uninstall).
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "git+https://github.com/NomanJafar/servellm" "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
# Private repos: wheels are embedded in the auto-generated cell above and written to /kaggle/working/wheels.
!pip install -q /kaggle/working/wheels/*.whl
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, inferchaos.slo, llmgateway.app, servellm.launch, tokenbench
assert torch.cuda.is_available()
import vllm.multimodal.media.audio
print("vllm", vllm.__version__, "torch", torch.__version__)

## A real replica as provider p0, a mock as p1

In [ ]:
import json, subprocess, time, httpx, threading
from servellm.launch import ServerConfig, VllmServer
from inferchaos.experiment import Experiment, Providers, _free_port, _wait_health
from inferchaos.slo import SLO

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
class RealAndMock(Providers):
    """p0 = vLLM on the GPU (killable, restartable with its real cold start); p1 = tokenbench mock fallback."""
    def __init__(self):
        super().__init__(n=2, capacity=16, ttft_ms=40, itl_ms=2)
        self.vllm = None; self.cold_starts = []
    def start(self):
        self._spawn("p1")
        self._start_vllm()
    def _start_vllm(self):
        t0 = time.monotonic()
        self.vllm = VllmServer(ServerConfig(model=MODEL, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9, port=8000),
                               log_path=f"results/startup_{len(self.cold_starts)}.log")
        self.vllm.start(ready_timeout_s=1200)
        self.cold_starts.append(time.monotonic() - t0)
        self.urls["p0"] = "http://127.0.0.1:8000"
    def kill(self, name):
        assert name == "p0"
        # run 3: SIGKILL on the API server alone left the EngineCore child holding 12.9 GiB, and the
        # restart found 1.6 GiB free. kill() now takes the whole process group; then wait until the
        # driver has actually released the memory before the restart is allowed to begin.
        self.vllm.kill()
        t0 = time.monotonic()
        for _ in range(120):
            used = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=20).stdout.split()[0])
            if used < 1024: break
            time.sleep(0.5)
        self.gpu_release_s = time.monotonic() - t0
    def restart(self, name):
        assert name == "p0"
        self._start_vllm()
    def stop(self):
        super().stop()
        if self.vllm: self.vllm.stop()

prov = RealAndMock(); prov.start()
print("vLLM cold start:", round(prov.cold_starts[0]), "s")

def warm(url, n=24):
    """Run 4 showed a freshly started vLLM answering its first requests in 3.5 s (Triton JIT of the attention and
    sampling kernels), which the breaker read as a sick provider. Warm the shapes the experiment uses before the
    baseline so the baseline measures the steady state; the restarted replica inside the experiment is deliberately
    NOT warmed, because its cold first requests are part of what a replica kill costs."""
    for i in range(n):
        httpx.post(f"{url}/v1/chat/completions", json={"model": MODEL, "messages": [{"role": "user", "content": "warm-up " * (8 + 40 * (i % 4))}],
                                                       "max_tokens": 32, "top_p": 0.9, "stream": False}, timeout=120)
t0 = time.monotonic(); warm("http://127.0.0.1:8000"); print(f"warm-up: {time.monotonic() - t0:.1f} s")

In [ ]:
slo = SLO(target=0.99, ttft_s=0.8)
exp = Experiment(prov, slo, rate=4.0, slo_ttft_ms=800, breaker_min_samples=10, breaker_window_s=15, breaker_cooldown_s=10, max_tokens=32)
# model name differs per provider: the gateway rewrites it from each ProviderConfig
exp.cfg.providers["p0"].model = MODEL
res = exp.run(fault="kill", baseline_s=60, fault_s=60, recovery_s=240)
res["vllm_cold_starts_s"] = prov.cold_starts; res["gpu_release_s"] = getattr(prov, "gpu_release_s", None)
prov.stop()
json.dump(res, open("results/kill_real_vllm.json", "w"), indent=1, default=str)
print({k: res[k] for k in ("requests", "errors", "fallbacks", "budget_consumed", "peak_burn_rate", "breaker_opened_at", "provider_restarted_at", "recovery_s")})
print("cold starts (s):", [round(c) for c in prov.cold_starts], "| alerts:", res["alerts"])

In [ ]:
from inferchaos.report import write_results
out = write_results({"kill (real vLLM on T4)": res}, "results")
print(out["table"])
from IPython.display import Image, display; display(Image("results/faults.png"))

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip